# Deep Learning

**Arquitetura:** _[MLP / CNN / RNN / Transformer]_  
**Dataset:** _[nome / fonte]_  
**Objetivo:** _[tarefa de aprendizado]_

## 1. Imports e Dispositivo

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED   = 42
torch.manual_seed(SEED)
print(f'Dispositivo: {DEVICE}')

## 2. Carregamento e Preparação dos Dados

In [ ]:
# Substitua pelos seus dados reais
# Exemplo sintético (classificação binária):
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=2000, n_features=20, random_state=SEED)
X = StandardScaler().fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

train_ds = TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train))
test_ds  = TensorDataset(torch.FloatTensor(X_test),  torch.LongTensor(y_test))

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=64)

print(f'Treino: {len(train_ds)} amostras | Teste: {len(test_ds)} amostras')

## 3. Definição do Modelo

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, num_classes: int):
        super().__init__()
        self.rede = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        return self.rede(x)


INPUT_DIM   = X_train.shape[1]
HIDDEN_DIM  = 128
NUM_CLASSES = 2

model = MLP(INPUT_DIM, HIDDEN_DIM, NUM_CLASSES).to(DEVICE)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parâmetros treináveis: {total_params:,}')

## 4. Configuração do Treino

In [ ]:
EPOCHS    = 30
LR        = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

## 5. Loop de Treino

In [ ]:
def run_epoch(loader, treino=True):
    if treino:
        model.train()
    else:
        model.eval()
    total_loss, corretos = 0.0, 0
    ctx = torch.no_grad() if not treino else torch.enable_grad()
    with ctx:
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            preds = model(X_batch)
            loss  = criterion(preds, y_batch)
            if treino:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(y_batch)
            corretos   += (preds.argmax(1) == y_batch).sum().item()
    return total_loss / len(loader.dataset), corretos / len(loader.dataset)


hist = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoca in range(1, EPOCHS + 1):
    tl, ta = run_epoch(train_loader, treino=True)
    vl, va = run_epoch(test_loader,  treino=False)
    scheduler.step()
    hist['train_loss'].append(tl); hist['val_loss'].append(vl)
    hist['train_acc'].append(ta);  hist['val_acc'].append(va)
    if epoca % 5 == 0:
        print(f'Época {epoca:3d} | Loss treino: {tl:.4f} | Acc treino: {ta:.4f} | Acc val: {va:.4f}')

## 6. Curvas de Aprendizado

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(hist['train_loss'], label='Treino')
ax1.plot(hist['val_loss'],   label='Validação')
ax1.set_title('Loss'); ax1.set_xlabel('Época'); ax1.legend()

ax2.plot(hist['train_acc'], label='Treino')
ax2.plot(hist['val_acc'],   label='Validação')
ax2.set_title('Acurácia'); ax2.set_xlabel('Época'); ax2.legend()

plt.tight_layout()
plt.show()

## 7. Avaliação Final

In [ ]:
from sklearn.metrics import classification_report

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        preds = model(X_batch.to(DEVICE)).argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(y_batch.numpy())

print(classification_report(all_labels, all_preds))

## 8. Conclusões

- _Performance final_
- _Overfitting/underfitting observado_
- _Próximos passos (regularização, arquitetura, dados)_